Customer Segmentation

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [2]:
df=pd.read_csv('C:\\Users\\USER\\Desktop\\data science\\customer_segmentation.csv')
df


,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,...,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response
0,5524,1957,Graduation,Single,58138.0,0,0,4/9/2012,58,635,...,7,0,0,0,0,0,0,3,11,1
1,2174,1954,Graduation,Single,46344.0,1,1,8/3/2014,38,11,...,5,0,0,0,0,0,0,3,11,0
2,4141,1965,Graduation,Together,71613.0,0,0,21-08-2013,26,426,...,4,0,0,0,0,0,0,3,11,0
3,6182,1984,Graduation,Together,26646.0,1,0,10/2/2014,26,11,...,6,0,0,0,0,0,0,3,11,0
4,5324,1981,PhD,Married,58293.0,1,0,19-01-2014,94,173,...,5,0,0,0,0,0,0,3,11,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2235,10870,1967,Graduation,Married,61223.0,0,1,13-06-2013,46,709,...,5,0,0,0,0,0,0,3,11,0
2236,4001,1946,PhD,Together,64014.0,2,1,10/6/2014,56,406,...,7,0,0,0,1,0,0,3,11,0
2237,7270,1981,Graduation,Divorced,56981.0,0,0,25-01-2014,91,908,...,6,0,1,0,0,0,0,3,11,0
2238,8235,1956,Master,Together,69245.0,0,1,24-01-2014,8,428,...,3,0,0,0,0,0,0,3,11,0


Feature Engineering

In [3]:
# Datetime conversion
df['Dt_Customer'] = pd.to_datetime(df['Dt_Customer'], format='mixed',dayfirst=True )

# Missing values(we have only 'Income' column with missing values)
df['Income'] = df['Income'].fillna(df['Income'].median())


# Feature engineering
from datetime import datetime
this_year = datetime.today().year
df['Age'] = this_year - df['Year_Birth']


spend_cols = ['MntWines','MntFruits','MntMeatProducts','MntFishProducts','MntSweetProducts','MntGoldProds']
df['Total_Spend'] = df[spend_cols].sum(axis=1)

# Family size caculation
df['Family_Size'] = df.apply(
    lambda x: 
        (1 if x['Marital_Status'] in ['Single', 'Alone', 'YOLO', 'Absurd', 'Divorced', 'Widow'] else 2)
        + x['Kidhome']
        + x['Teenhome'],
    axis=1
)


In [4]:
df.groupby('Education')['Total_Spend'].mean().sort_values()
# Replace 'Graduation' with 'Graduated' for consistency
df['Education'] = df['Education'].replace({'Graduation': 'Graduated'})

Preparing for clustering

In [5]:
#features
features = ['Income','Age','Recency','Total_Spend','NumWebPurchases','NumStorePurchases']
# target
targets = df['Response']

Modelling for clustering

In [6]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans


X = df[features]
X_scaled = StandardScaler().fit_transform(X)


kmeans = KMeans(n_clusters=4, random_state=42)
df['Segment'] = kmeans.fit_predict(X_scaled)

c:\Users\USER\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=9.
  warnings.warn(


In [7]:
# to see which rows belong to which cluster
df['Cluster'] = kmeans.labels_

# Create separate DataFrames for each segment

segment_0 = df[df['Cluster'] == 0]
segment_1 = df[df['Cluster'] == 1]
segment_2 = df[df['Cluster'] == 2]
segment_3 = df[df['Cluster'] == 3]
segment_0

,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,...,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response,Age,Total_Spend,Family_Size,Segment,Cluster
1,2174,1954,Graduated,Single,46344.0,1,1,2014-03-08,38,11,...,0,0,3,11,0,72,27,3,0,0
3,6182,1984,Graduated,Together,26646.0,1,0,2014-02-10,26,11,...,0,0,3,11,0,42,53,3,0,0
7,6177,1985,PhD,Married,33454.0,1,0,2013-05-08,32,76,...,0,0,3,11,0,41,169,3,0,0
8,4855,1974,PhD,Together,30351.0,1,0,2013-06-06,19,14,...,0,0,3,11,1,52,46,3,0,0
10,1994,1983,Graduated,Married,51381.5,1,0,2013-11-15,11,5,...,0,0,3,11,0,43,19,3,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2222,10659,1979,2n Cycle,Together,7500.0,1,0,2013-05-07,7,2,...,0,0,3,11,0,47,53,3,0,0
2223,1448,1963,Master,Married,33562.0,1,2,2014-06-25,33,21,...,0,0,3,11,0,63,51,5,0,0
2229,10084,1972,Graduated,Married,24434.0,2,0,2014-05-18,9,3,...,0,0,3,11,0,54,50,4,0,0
2232,8080,1986,Graduated,Single,26816.0,0,0,2012-08-17,50,5,...,0,0,3,11,0,40,22,1,0,0


In [8]:
segment_interpretation = pd.DataFrame({
    'Segment': [0, 1, 2, 3],
    'Segment_Name': [
        'High-value loyal customers',
        'Low-spend, high-visit browsers',
        'Digital-first customers',
        'Price-sensitive families'
    ],
    'Description': [
        'High spending, frequent purchases, strong loyalty and campaign response',
        'Frequent visits but low spending and weak conversion',
        'Strong preference for online channels and digital engagement',
        'Larger households with strong sensitivity to price and promotions'
    ],
    'Recommended_Action': [
        'VIP programs, loyalty rewards, personalized premium offers',
        'Discount nudges, entry-level bundles, retargeting campaigns',
        'Email/app-first campaigns, online exclusives',
        'Family bundles, bulk discounts, value packs'
    ]
})
segment_interpretation

,Segment,Segment_Name,Description,Recommended_Action
0,0,High-value loyal customers,"High spending, frequent purchases, strong loya...","VIP programs, loyalty rewards, personalized pr..."
1,1,"Low-spend, high-visit browsers",Frequent visits but low spending and weak conv...,"Discount nudges, entry-level bundles, retarget..."
2,2,Digital-first customers,Strong preference for online channels and digi...,"Email/app-first campaigns, online exclusives"
3,3,Price-sensitive families,Larger households with strong sensitivity to p...,"Family bundles, bulk discounts, value packs"


Modeling with RandomForestClassifier

In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, targets, test_size=0.2, random_state=42)
rf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)
rf.fit(X_train, y_train)
rf.score(X_test, y_test)

0.8526785714285714

Modelling with Decision Tree

In [10]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
import pandas as pd

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, targets, test_size=0.2, random_state=42)

# Initialize Decision Tree
dt = DecisionTreeClassifier(max_depth=6, random_state=42)

# Train the model
dt.fit(X_train, y_train)

# Evaluate accuracy
accuracy = dt.score(X_test, y_test)
print(f"Decision Tree Accuracy: {accuracy:.4f}")

# Feature importance
feature_importances = pd.Series(dt.feature_importances_, index=X.columns).sort_values(ascending=False)
print("Feature Importances:")
print(feature_importances)

# Confusion matrix
y_pred = dt.predict(X_test)
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)


Decision Tree Accuracy: 0.8460
Feature Importances:
Total_Spend          0.423323
Recency              0.244569
NumStorePurchases    0.106476
Income               0.088892
Age                  0.076134
NumWebPurchases      0.060604
dtype: float64
Confusion Matrix:
[[364  15]
 [ 54  15]]
